# 演習7. パイプラインを組む

## シーン

部品はすべてそろいました。

- スレッドを起動して `join()` する（演習1）
- 共有データを鍵で守る（演習2・3）
- 相手を待つ（演習4）
- 容量つきのスレッドセーフなキュー（演習5・6）

この演習では、これらを使って **3段のパイプラインを組んで、測ります。**

```
[ Read ] --q1--> [ Infer ] --q2--> [ Show ]
```

**各段は「計算する時間」と「待つ時間」の両方を持たせます。** 本番がそうだからです。

```
              計算      待ち      合計
Read          10ms       0ms      10ms      デコードは CPU の仕事
Infer         25ms   0/35/70ms  25〜95ms    前後処理は CPU、DPU に投げている間は待ち
Show          12ms      18ms      30ms      描画や送信は待ち寄り
```

- **Infer だけ時間がばらつく**のは、現実がそうだからです。写っているものが多ければ重くなります
- **1フレームの CPU 総量は 10 + 25 + 12 = 47ms**。ここは段をどう並べても減りません

演習1で見たとおり、**待つ仕事はコア数に関係なく重なりますが、計算する仕事は重なりません。**
そして Colab の `hardware_concurrency() = 2` は、**計算に関しては当てになりませんでした。**
この演習は、そこも含めて一度に測ります。

### 測るもの ―― 全部 FPS に直します

プログラムがやることは1つだけです。

> **30フレームを流して、30枚目が Show を出終わるまでの時間を測る。**

その時間を、そのままでは表に出しません。**1秒あたり何枚さばけたか**に直します。

```
FPS = 流したフレーム数 / かかった秒数
```

- **30枚が 1.2秒 で終わった ⇒ 30 / 1.2 = 25.0 FPS**
- **30枚が 3.0秒 で終わった ⇒ 30 / 3.0 = 10.0 FPS**

30枚というのはこの演習の都合で決めた数で、意味はありません。
**FPS に直せば、何枚流したかに関係なく比べられます。**

かかった時間そのものは表に出しません。**ms が出てくるのは表の2か所だけ**です。

- **最遅段** … 1フレームあたりの、一番遅い段の時間
- **1枚の遅れ** … 1枚が入ってから出るまでの時間（レイテンシ）

この2つは「1枚あたり」の量で、FPS とは別の話です。**混ぜないでください。**

## 7-1. 【予測クイズ】段に分けて、人を増やしていく

次のプログラムは、**5つの構成を続けて測ります。**

```
(1) 直列（1本の for ループ。キューもスレッドも使わない）
(2) 3段パイプライン（Read / Infer / Show を1人ずつ）
(3) (2) の Infer を 2人に
(4) (2) の Infer を 3人に
(5) Infer 2人 + Show も 2人
```

キューは演習5で作ったものをそのまま使います（容量4）。
コードの上のほうにキューのクラスが載っていますが、**読み飛ばして構いません。**
見るべきは各段のスレッドの形だけです。

```cpp
std::thread reader ([&] { for (...) { 読む;              q1.push(f); } });
std::thread inferer([&] { for (...) { f = q1.pop(); 推論; q2.push(f); } });
std::thread shower ([&] { for (...) { f = q2.pop(); 表示;             } });
```

**きれいに「取り出す → 処理する → 入れる」の繰り返しになっている**ことに注目してください。
これがパイプラインの1段の形です。人を2人にするときも変えるのはループの回し方だけで、
**2人とも同じ `q1` から取り出し、同じ `q2` に入れます。**
キューがそのまま仕事の配り口になるので、分担のコードは要りません。

**実行する前に、5つとも予測してください。**

- (1) は 1フレーム 100ms なので 10 FPS のはずです。ここが基準です
- (2) は3段に分けたのだから、3倍の 30 FPS になるでしょうか
- (3) で一番遅い段（Infer）を2人にしたら、(2) の2倍になるでしょうか
- (4) で3人にしたら、さらに伸びるでしょうか
- (5) で Show も2人にしたら、どうなるでしょうか

**もう1つ予測してほしいことがあります。** Show に届くフレームの**順番**はどうなるでしょうか。

なお、各段のスレッドが必ずこの形になることは覚えておいてください。

```cpp
while (まだある) {
    T x = 前のキュー.pop();     // 取り出す（空なら待つ）
    ...処理する...              // ここは鍵の外。ここだけが本当の仕事
    次のキュー.push(x);         // 入れる（満杯なら待つ）
}
```

**待つ・鍵をかけるは、全部キューの中に閉じ込められています。**
段のコードには `mutex` も `condition_variable` も出てきません。これが演習5の成果です。

In [ ]:
%%writefile ex07a.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <vector>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 演習5で組み立てたキュー（中身は読まなくてよい） ----
template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v); lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop(); lk.unlock(); can_push_.notify_one();
        return v;
    }
private:
    std::queue<T> q_;
    std::size_t capacity_;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

// ================= 仕事の中身 =================
// 各段は「計算する時間」と「待つ時間」の2つを持つ。本番もそうなっている。
//                        計算  待ち   合計
const int READ_CPU = 10, READ_WAIT = 0;              //  10ms  デコードは計算
const int INF_CPU  = 25;                             //  待ちは 0/35/70（平均 35）
const int SHOW_CPU = 12, SHOW_WAIT = 18;             //  30ms  描画・送信は待ち寄り

const int N = 30;
const std::size_t CAP = 4;

const double CPU_PER_FRAME  = READ_CPU + INF_CPU + SHOW_CPU;            // 47ms
const double READ_MS = READ_CPU + READ_WAIT;                            // 10ms
const double INF_MS  = INF_CPU + 35;                                    // 60ms（平均）
const double SHOW_MS = SHOW_CPU + SHOW_WAIT;                            // 30ms

long   calib = 0;                 // 1ms 分の繰り返し回数
volatile long sink = 0;

long burn(long n) { long s = 0; for (long i = 0; i < n; i++) s += (i * 2654435761u) % 7; return s; }

void calibrate() {
    long n = 100000;
    for (;;) {
        auto t0 = steady_clock::now();
        sink += burn(n);
        auto us = duration_cast<microseconds>(steady_clock::now() - t0).count();
        if (us > 30000) { calib = n * 1000 / us; break; }
        n *= 2;
    }
}

// hardware_concurrency() を信じず、「同時にどれだけ計算が進むか」を実測する
double effective_cores(unsigned hw) {
    long U = calib * 60;
    auto t0 = steady_clock::now();
    sink += burn(U);
    double one = duration_cast<microseconds>(steady_clock::now() - t0).count();
    std::vector<std::thread> th;
    t0 = steady_clock::now();
    for (unsigned k = 0; k < hw; k++) th.emplace_back([&]{ sink += burn(U); });
    for (auto& t : th) t.join();
    double many = duration_cast<microseconds>(steady_clock::now() - t0).count();
    return hw * one / many;
}

void stage(int cpu_ms, int wait_ms) {
    if (cpu_ms)  sink += burn(calib * cpu_ms);
    if (wait_ms) std::this_thread::sleep_for(milliseconds(wait_ms));
}
void do_read()          { stage(READ_CPU, READ_WAIT); }
void do_infer(int i)    { stage(INF_CPU, (i % 3) * 35); }
void do_show()          { stage(SHOW_CPU, SHOW_WAIT); }

struct Result { int ms; double lat; int inv; };

Result serial() {
    double lat = 0;
    auto t0 = steady_clock::now();
    for (int i = 0; i < N; i++) {
        auto born = steady_clock::now();
        do_read(); do_infer(i); do_show();
        lat += duration_cast<milliseconds>(steady_clock::now() - born).count();
    }
    return { (int)duration_cast<milliseconds>(steady_clock::now() - t0).count(), lat / N, 0 };
}

Result pipeline(int ninfer, int nshow) {
    struct Item { int idx; steady_clock::time_point born; };
    BoundedQueue<Item> q1(CAP), q2(CAP);
    double lat = 0; int inv = 0, last = -1;
    std::mutex m;

    auto t0 = steady_clock::now();
    std::thread rd([&]{ for (int i = 0; i < N; i++) { do_read(); q1.push({i, steady_clock::now()}); } });

    std::vector<std::thread> inf;
    for (int k = 0; k < ninfer; k++)
        inf.emplace_back([&]{
            for (int i = k; i < N; i += ninfer) { Item it = q1.pop(); do_infer(it.idx); q2.push(it); }
        });

    std::vector<std::thread> sh;
    for (int k = 0; k < nshow; k++)
        sh.emplace_back([&]{
            for (int i = k; i < N; i += nshow) {
                Item it = q2.pop();
                do_show();
                std::lock_guard<std::mutex> g(m);
                if (it.idx < last) inv++;
                last = it.idx;
                lat += duration_cast<milliseconds>(steady_clock::now() - it.born).count();
            }
        });

    rd.join(); for (auto& t : inf) t.join(); for (auto& t : sh) t.join();
    return { (int)duration_cast<milliseconds>(steady_clock::now() - t0).count(), lat / N, inv };
}

double fps_cores = 0;   // 上限B（コア数で決まる。どの構成でも同じ）

// 数字を先に、日本語のラベルを最後に置く（setw はバイト数で数えるため）
void line(const char* label, double slowest_stage, Result r) {
    std::cout << std::fixed << std::setprecision(1);
    if (slowest_stage > 0) std::cout << std::setw(6) << std::setprecision(0) << slowest_stage << "ms"
                                     << std::setw(8) << std::setprecision(1) << (1000.0 / slowest_stage);
    else                   std::cout << std::setw(8) << "-" << std::setw(8) << "-";
    std::cout << std::setw(8) << fps_cores
              << std::setw(8) << (1000.0 * N / r.ms)
              << std::setw(8) << std::setprecision(0) << r.lat << "ms"
              << std::setw(6) << r.inv
              << "   " << label << "\n";
}

int main() {
    unsigned hw = std::thread::hardware_concurrency();
    calibrate();
    double eff = effective_cores(hw);
    fps_cores = 1000.0 * eff / CPU_PER_FRAME;

    std::cout << std::fixed;
    std::cout << N << "フレームを流し、全部が Show を出終わるまでの時間を測ります。\n"
              << "FPS = " << N << "フレーム / かかった秒数   （1.2秒で終われば 25.0 FPS）\n"
              << "以下の数字は「遅れ」以外すべて FPS（1秒あたりの枚数）です。\n\n";

    std::cout << "1フレームの CPU 総量 = " << std::setprecision(0)
              << READ_CPU << " + " << INF_CPU << " + " << SHOW_CPU << " = " << CPU_PER_FRAME << "ms\n";
    std::cout << "hardware_concurrency() = " << hw
              << " ですが、同時に進む計算の量を実測すると " << std::setprecision(2) << eff << " 個分でした\n";
    std::cout << "\n上限A = 1000 / 一番遅い段の時間                      ... 段の分け方と人数で動く\n";
    std::cout << "上限B = 1000 x " << std::setprecision(2) << eff << " / "
              << std::setprecision(0) << CPU_PER_FRAME << "ms = "
              << std::setprecision(1) << fps_cores
              << " FPS                  ... 計算量を減らさない限り動かない\n\n";

    std::cout << "最遅段     上限A   上限B 実測FPS 1枚の遅れ  乱れ   構成\n";
    line("直列（1本）", 0, serial());
    line("3段", INF_MS, pipeline(1, 1));
    line("3段 + Infer 2人", SHOW_MS, pipeline(2, 1));
    line("3段 + Infer 3人", SHOW_MS, pipeline(3, 1));
    line("3段 + Infer 2人 + Show 2人", READ_MS, pipeline(2, 2));
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread -O2 ex07a.cpp -o ex07a && ./ex07a

## 7-2. 結果

**下に載せたのは、ある1回の実行結果です。** 実行ごとに1〜2割ぶれますし、
`1.68 個分` の数字はマシンによって変わります。この数字が大きいマシンほど
上限B が上に行き、下の3行が伸びます。**数字そのものではなく、次の3つを見てください。**

- **実測FPS が、上限A と 上限B のどちらに近いか**
- **上限A を上げたとき、実測が付いてくるかどうか**
- **速くなったとき、遅れと乱れがどうなっているか**

```
30フレームを流し、全部が Show を出終わるまでの時間を測ります。
FPS = 30フレーム / かかった秒数   （1.2秒で終われば 25.0 FPS）
以下の数字は「遅れ」以外すべて FPS（1秒あたりの枚数）です。

1フレームの CPU 総量 = 10 + 25 + 12 = 47ms
hardware_concurrency() = 2 ですが、同時に進む計算の量を実測すると 1.68 個分でした

上限A = 1000 / 一番遅い段の時間                      ... 段の分け方と人数で動く
上限B = 1000 x 1.68 / 47ms = 35.7 FPS                  ... 計算量を減らさない限り動かない

最遅段     上限A   上限B 実測FPS 1枚の遅れ  乱れ   構成
       -       -    35.7     9.6     103ms     0   直列（1本）
    60ms    16.7    35.7    16.0     345ms     0   3段
    30ms    33.3    35.7    24.2     301ms     8   3段 + Infer 2人
    30ms    33.3    35.7    26.9     334ms     9   3段 + Infer 3人
    10ms   100.0    35.7    23.5     237ms     7   3段 + Infer 2人 + Show 2人
```

### 7-2-1. まず、2つの上限を押さえる

表の見方はこれだけです。**実測FPS は、上限A と 上限B の小さいほうを超えられません。**

**上限A ―― 一番遅いスレッドが効いてくる**

```
上限A = 1000 / 一番遅い段の時間(ms)
```

流れ作業なので、**全体の速さは一番遅い係の速さ**になります。
だから **段の切り方と、各段の人数を変えれば動きます。**
一番遅い段を2人にすれば、その段は実質半分の時間になり、上限A は上がります。

**上限B ―― CPU のコア数が効いてくる**

```
上限B = 1000 x 同時に進む計算の量 / 1フレームの CPU 総量(ms)
```

こちらは **1秒間にこのマシンが実行できる計算量**そのものです。
**式にスレッドの本数も段の数も出てきません。** 人を何人並べようが、
1フレームに必要な計算量（10 + 25 + 12 = 47ms）は1ミリも減らないからです。

> **人を増やして減るのは「待ち時間」であって、「計算量」ではない。**

上の表で **上限B の列が全部 35.7 で同じ**なのは、そういうわけです。

### 7-2-2. 段に分けても、3倍にはならない（上限A の話）

直列 9.6 FPS → 3段 16.0 FPS。**1.65倍**です。3倍ではありません。

このとき最遅段は Infer の 60ms。上限A は `1000 / 60 = 16.7 FPS` で、
**実測 16.0 はほぼ上限A に張り付いています。**

```
Read   R.........R.........R.........       10ms、ほとんど手待ち
Infer  IIIIIIIIIIIIIIIIIIIIIIIIIIIIII       60ms、働きづめ
Show   SSS.......SSS.......SSS.......       30ms、半分手待ち
```

> **段に分けて得られる倍率は「1フレームの合計時間 ÷ 一番遅い段の時間」まで。**

### 7-2-3. 人を増やすと上限A は上がる。しかし実測は上限B で止まる

Infer を2人にすると最遅段が Show（30ms）に移り、**上限A は 16.7 → 33.3 に上がります。**
実測も 16.0 → 24.2 と伸びました。

ところが **3人にしても 26.9。誤差の範囲でしか動きません**（下がることもあります）。
上限A は 33.3 のままなので、**上限A では説明が付きません。**

説明が付くのは上限B のほうです。**35.7 FPS。**
実測 24〜27 は、その7割あたりで頭打ちになっています。

なお **実測が上限に届ききらない**のは普通です。キューの出し入れ、スレッドの入れ替え、
キャッシュの奪い合いの分だけ必ず目減りします。
**上限は「これより速くはならない」という天井**であって、達成目標ではありません。

**マシンによっては、もっとはっきり出ます。** 「同時に進む計算の量」が 1.0 に近い環境では
上限B は 21 FPS 前後まで下がり、上限A（33.3）よりはっきり小さくなります。
そうなると Infer を2人にしても3人にしても 21 FPS 前後で完全に止まります。

### 7-2-4. `hardware_concurrency()` は当てにならない

出力の行に注目してください。

```
hardware_concurrency() = 2 ですが、同時に進む計算の量を実測すると 1.68 個分でした
```

この関数が返すのは **OS から見えるスレッドの数**であって、
**同時に計算できる回路の数ではありません。** Colab の「2」は、
実際には計算回路1つ分に近い性能しかありません（演習1で測ったとおりです）。

だから上限B を計算するときに `hardware_concurrency()` をそのまま使ってはいけません。
このプログラムは、**同じ計算を1本で回したときと全部同時に回したときの速度比を実測**して、
その値を使っています。

```cpp
// 1本で burn(U) → one マイクロ秒 ; hw本同時に burn(U) → many マイクロ秒
return hw * one / many;      // これが「同時に進む計算の量」
```

> **数えるのではなく、測る。**

### 7-2-5. Show も2人にしても、もう伸びない

これが一番はっきりした結果です。

```
3段 + Infer 2人              上限A  33.3   実測 24.2
3段 + Infer 2人 + Show 2人   上限A 100.0   実測 23.5   ← 上限A は3倍、実測は横ばい
```

Show を2人にすると、上限A は 33.3 → 100.0 と**3倍に跳ね上がります。**
それなのに実測は動きません。**環境によっては、はっきり下がります。**

**効いていないほうの上限を上げても、何も起きない**からです。
上限B で止まっているところに5本目のスレッドを足したので、
得るものは何もなく、切り替えの手間だけが増えました。

> **効いていない上限を上げる改造は、良くて無駄、悪ければ損。**

（そもそも本番の Show は画面や送信先が1つしかないので、
2人にしても順番待ちになるだけでした ―― 演習1の発展課題4）

### 7-2-6. 速くすると、レイテンシと順序を失う

**1枚の遅れ**の列を見てください。1フレームが Read を出てから画面に出るまでの時間です。

```
直列              103ms      ← 一番よい
3段               345ms
Infer 2人         301ms
```

FPS は2.5倍になったのに、**遅れは3倍近く悪くなっています。**
キューで順番待ちをするぶんが増えたからです。演習1で書いたとおり、
**スレッドが改善するのはスループット側だけ**です。

**乱れ**の列は、Show に届いたフレームの番号が前より小さかった回数です。

```
直列          0
3段           0
Infer 2人     8      ← 追い越しが起きている
Infer 3人     9
```

原因は単純です。

```
Infer係A   [ フレーム2 を推論 95ms ]
Infer係B   [ 3を推論 25ms ]  [ 4を... ]
q2 に入る順  3, 4, 2, ...     ← 追い越しが起きる
```

これは**バグではありません。** 並列化とは、そういうことです。
動画の表示ならガタガタして見えますし、検出結果とフレーム番号を対応づけていたら、
**別のフレームに枠が付きます。この直し方が演習8のテーマです。**

## 7-3. では、どう速くしていくのか

ここからが本題です。**本番のプログラムを前にして、何をどの順で変えるか。**

### 7-3-1. 手順

```
[0] 基準を記録する
       いまの FPS ／ 1枚の遅れ ／ 順序の乱れ を控える
       ここを取らずに改造を始めない
         |
[1] 各段の時間を測る（演習10）
       各段について「正味」と「待ち」を分けて出す
         |
[2] 2つの上限を計算する
       上限A = 1000 / 一番遅い段の時間
       上限B = 1000 x 同時に進む計算の量 / 各段の「正味」の合計
         |
[3] 実測FPS を、この2つと見比べる
         |
    +----------------+----------------+
    |                |                |
実測 ≒ 上限A     実測 ≒ 上限B     どちらにも遠い
（遅い段で         （CPU で          （構成の問題）
  詰まっている）     詰まっている）
    |                |                |
一番遅い段を      計算量を減らす    キューの容量、
2人にする or      ・入力を小さく    段の切り方、
その段を切り分ける ・軽いモデル      出入りの回数
    |             ・間引く          を疑う
    |             ・重い表示をやめる     |
    +----------------+----------------+
                     |
[4] 1回に1つだけ変えて、[0] に戻る
```

**分岐は3つだけです。** この3つを取り違えなければ、あとは繰り返すだけになります。

### 7-3-2. 動かせるパラメータと、それが動かすもの

どのつまみを回すと何が動くのか、先に整理しておきます。

**段の切り方（何段にするか）**

- 上限A が動きます。細かく切るほど上がります
- **上限B は動きません。**切っても計算の総量は変わらないからです
- 1枚の遅れは**悪くなります**（通る待ち行列が増えるため）

**各段の人数**

- **その段が最遅段のときだけ**上限A が動きます。それ以外は損（7-2-5）
- 上限B は動きません
- 順序が**崩れます**

**キューの容量**

- **上限はどちらも動きません**
- 変わるのは「でこぼこの吸収力」だけ（演習6、発展課題2）
- 大きくすると1枚の遅れとメモリが増えます

**間引き（N枚に1枚だけ表示する）**

- **上限B が上がります**（表示の計算をしなくなるため）
- ただし**画面に出る枚数は増えません**。増えるのは処理側の FPS だけです

**重い段そのものを軽くする（表示方法を変える、解像度を下げる、軽いモデルにする）**

- **上限A も 上限B も同時に上がります**
- **効きが一番大きいのはここ**です。imshow をやめて圧縮してPCへ送るような変更が該当します

> **上限A はスレッドの並べ方で動く。上限B は仕事を減らさない限り動かない。**

### 7-3-3. 本番では、どちらが効いているのか

同じ `yolov3_video_study.cpp` でも、**モデルによって答えが変わります。**

```
yolov3 : DPU が 82ms 中 67ms。そこは「待ち」なので、正味の CPU は 20ms 程度
         上限B = 1000 x 4コア / 20ms = 200 FPS
         上限A = 1000 / 67ms         =  15 FPS   ← こちらが効いている

tiny   : DPU がわずか 2ms。正味の CPU は 10ms 程度
         上限B = 1000 x 4コア / 10ms = 400 FPS
         上限A は表示の段が決める                ← まだ A だが、差は縮んでいる
```

**yolov3 なら、やることは1つです。**「DPU 待ちの 67ms を短くする」以外に効く手はありません。
スレッドを増やしても、キューをいじっても、上限A は動きません。

**モデルを軽くするほど、上限B が効いてくる方向に近づきます。**
「DPU を速くしたのに全体が速くならない」というときは、上限B を疑ってください。

### 7-3-4. やってはいけない3つ

**① ボトルネックでない段の人を増やす** ―― 7-2-5 で見たとおり、遅くなります。

**② 一度に2つ変える** ―― どちらが効いたのか永久に分かりません。
片方が +20%、もう片方が -20% なら、変化なしに見えます。

**③ FPS だけ見る** ―― 遅れと順序は FPS と引き換えに悪くなります。
**毎回3つとも記録してください。** 本番の1行出力がその形になっています。

### 7-3-5. 止めどき

次のどれかで止めます。

- **要求している FPS に届いた**
- **実測が上限（小さいほう）に十分近づいた** ―― 残っているのは数％です
- **変えても FPS が上がらなくなった** ―― [3] の見立てが違っています。[1] からやり直し

> **測る → 効いている上限を見分ける → そこだけ動かす → また測る。**

## 発展課題

1. Read（10ms）を2人にしたらどうなるでしょうか。予測してから、理由を説明してください。

2. キューの容量を 4 から **1** に変えたら、7-2 の FPS はどうなるでしょうか。
   Infer の所要時間が 25 / 60 / 95ms とばらついていることに注目してください。

3. 段の切り方を変えて、**「Read+Infer」で1段、「Show」で1段**の2段パイプラインにしたら、
   FPS はどうなるでしょうか。スレッドは2本、キューは1本になります。

4. 7-2 の順番の崩れを直すには、どんな方法が考えられるでしょうか。
   **コードを書く前に、案を2つ以上出してみてください。**
   （演習8で答え合わせをします）